# Actividad: Predicción de Precios de Vivienda (Modelos de Regresión)

## Maestría en Inteligencia Artificial Aplicada

**Tecnológico de Monterrey**

**Curso:** Inteligencia Artificial y Aprendizaje Automático

**Profesor:** Luis Eduardo Falcón Morales

**Nombre:** Carlos Rodrigo Salguero Alcántara

**Matrícula:** A00833341


## 1.0 Introducción

El mercado inmobiliario combina decenas de atributos de la propiedad (tamaño, calidad, ubicación, antigüedad) con condiciones de mercado. Aquí el objetivo no es ganar el leaderboard de Kaggle tal cual, sino **valuar de forma estática** el precio de cierre `SalePrice`: sin tendencias cronológicas absolutas, con un flujo reproducible y comparable entre modelos.

Dataset: [House Prices – Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) (`data/train.csv`). El `test.csv` de Kaggle no trae la etiqueta, así que la partición train/test y la validación cruzada se hacen sobre `train.csv`.


### Hoja de ruta: dónde se responde cada punto de la actividad

| Pregunta / entrega | Sección |
| --- | --- |
| 1. Decisión sobre `YrSold` | 4.1 |
| 2. `YearBuilt` / `YearRemodAdd` → edades | 4.2 |
| 3. Decisión sobre `MoSold` | 4.3 |
| 4–5. Limpieza, imputación; `PoolQC` y `GarageType` | 5.0 |
| 6. One-Hot y el hiperparámetro `drop` | 6.0 |
| 7. Partición train / test (+ CV) | 7.0 |
| 8. Modelo base (ingenuo) | 8.0 |
| 9a–9f. Modelos + curvas de aprendizaje + tabla | 9.0 |
| 10. Análisis de factores | 10.0 |
| 11. Top-3 con RMSE + MAE | 11.0 |
| 12–15. Voting, Stacking y comparación | 12.0 |
| 16. Reportes técnico y ejecutivo | 13.0 |
| 17. Conclusiones y uso de LLMs | 14.0 |


## 2.0 Configuración del notebook

Importaciones, constantes tipadas y estilo de gráficas. `QUICK=True` acorta las búsquedas para verificar que todo corre; para la entrega final usar `QUICK=False`.


### 2.1 Importaciones


In [ ]:
from __future__ import annotations

import time
import warnings
from pathlib import Path
from typing import Any, Final


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import Lasso, LinearRegression, Ridge, RidgeCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    cross_val_predict,
    learning_curve,
    train_test_split,
)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor


### 2.2 Constantes y estilo


In [ ]:
warnings.filterwarnings("ignore")

SEED: Final[int] = 42
CURRENT_YEAR: Final[int] = 2026
N_JOBS: Final[int] = -1
QUICK: Final[bool] = False  # True = búsquedas mínimas para smoke-test
TEST_SIZE: Final[float] = 0.20
CV_SPLITS: Final[int] = 5

DATA_PATH: Final[Path] = Path("data") / "train.csv"
TARGET_COL: Final[str] = "SalePrice"

PLOT_PRIMARY: Final[str] = "#2F6F8F"
PLOT_SECONDARY: Final[str] = "#C46B3A"
PLOT_ACCENT: Final[str] = "#3D8B6E"
PLOT_EDGE: Final[str] = "white"


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)


In [ ]:
def style_axes(ax: plt.Axes, title: str, xlabel: str = "", ylabel: str = "") -> plt.Axes:
    ax.set_title(title, pad=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return ax


def finalize_figure() -> None:
    plt.tight_layout()
    plt.show()


def rmse(y_true: np.ndarray | pd.Series, y_pred: np.ndarray) -> float:
    """Root mean squared error in the same units as y (USD here)."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


## 3.0 Carga de datos

Se prioriza `data/train.csv` (layout del repositorio). Si no está, se intenta `train.csv` en el cwd; en Colab, upload manual.


In [ ]:
def load_house_prices(path: Path = DATA_PATH) -> pd.DataFrame:
    """
    Load Kaggle train.csv. Fail with a clear message if the file is missing.
    """
    candidates = [path, Path("train.csv"), Path("data") / "raw" / "train.csv"]
    for candidate in candidates:
        if candidate.exists():
            df = pd.read_csv(candidate)
            print(f"Datos cargados desde: {candidate.resolve()}")
            return df

    try:
        from google.colab import files

        uploaded = files.upload()
        return pd.read_csv(next(iter(uploaded)))
    except Exception as exc:  # noqa: BLE001
        raise FileNotFoundError(
            f"No se encontró {path}. Coloca train.csv en data/."
        ) from exc


In [ ]:
df_raw: pd.DataFrame = load_house_prices()
print(f"Shape: {df_raw.shape}")
print(f"Memory: {df_raw.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")
df_raw.head()


In [ ]:
miss = df_raw.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
display(
    pd.DataFrame({"faltantes": miss, "pct": (miss / len(df_raw) * 100).round(2)}).head(20)
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(df_raw[TARGET_COL], kde=True, ax=axes[0], color=PLOT_PRIMARY)
style_axes(axes[0], "SalePrice (asimétrica)", "USD", "Count")
sns.histplot(np.log1p(df_raw[TARGET_COL]), kde=True, ax=axes[1], color=PLOT_SECONDARY)
style_axes(axes[1], "log1p(SalePrice)", "log1p(USD)", "Count")
finalize_figure()
print(
    f"Asimetría SalePrice: {df_raw[TARGET_COL].skew():.2f} | "
    f"log1p: {np.log1p(df_raw[TARGET_COL]).skew():.2f}"
)


## 4.0 Decisiones sobre variables temporales

La actividad pide una valuación **estática**: sin tendencias cronológicas absolutas.


### 4.1 `YrSold` — se elimina

`YrSold` solo toma unos pocos años de calendario (2006–2010). En un plano estático no aporta una característica intrínseca de la vivienda: captura coyuntura (p. ej. 2008). Incluirla contradice la restricción del enunciado y no extrapolable a un “presente” genérico. **Decisión: eliminar.**


### 4.2 `YearBuilt` / `YearRemodAdd` → `AgeHouse` / `AgeRemod`

Se convierten a edades relativas a **2026**:

- `AgeHouse = 2026 − YearBuilt`
- `AgeRemod = 2026 − YearRemodAdd`

Una edad se interpreta directo en valuación (“casa de 20 años”). **Las originales se eliminan**: son una transformación lineal exacta de las edades → colinealidad perfecta en modelos lineales y cero información nueva. El mismo criterio aplica a `GarageYrBlt` → `AgeGarage` (si no hay garaje queda NaN y se imputa *dentro* del pipeline con indicador de faltante).


### 4.3 `MoSold` — seno / coseno (cíclica)

El mes es circular: diciembre ≈ enero. Como numérica impone distancia 12↔1 falsa; como One-Hot agrega muchas columnas y pierde cercanía. Uso:

\[
MoSold\_sin = \sin(2\pi \cdot MoSold / 12),\quad
MoSold\_cos = \cos(2\pi \cdot MoSold / 12)
\]

**Se elimina el original** porque queda representado por el par (sin, cos) y mantenerlo reintroduce el orden lineal artificial.


In [ ]:
# Atípicos documentados por el autor del dataset (casas enormes vendidas muy baratas)
n_outliers = int(((df_raw["GrLivArea"] > 4000) & (df_raw[TARGET_COL] < 300_000)).sum())
print(f"Atípicos GrLivArea>4000 y precio bajo a eliminar: {n_outliers}")

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=df_raw, x="GrLivArea", y=TARGET_COL, alpha=0.45, ax=ax, color=PLOT_PRIMARY
)
ax.axvline(4000, color=PLOT_SECONDARY, ls="--", lw=1.2)
style_axes(ax, "GrLivArea vs SalePrice", "GrLivArea", "SalePrice (USD)")
finalize_figure()


In [ ]:
df: pd.DataFrame = (
    df_raw[~((df_raw["GrLivArea"] > 4000) & (df_raw[TARGET_COL] < 300_000))]
    .reset_index(drop=True)
    .copy()
)

df["AgeHouse"] = CURRENT_YEAR - df["YearBuilt"]
df["AgeRemod"] = CURRENT_YEAR - df["YearRemodAdd"]
df["AgeGarage"] = CURRENT_YEAR - df["GarageYrBlt"]  # NaN = sin garaje
df["MoSold_sin"] = np.sin(2 * np.pi * df["MoSold"] / 12)
df["MoSold_cos"] = np.cos(2 * np.pi * df["MoSold"] / 12)

df = df.drop(
    columns=["Id", "YrSold", "YearBuilt", "YearRemodAdd", "GarageYrBlt", "MoSold"]
)
df["MSSubClass"] = df["MSSubClass"].astype(str)

print(f"Shape tras transformaciones temporales: {df.shape}")
df[["AgeHouse", "AgeRemod", "MoSold_sin", "MoSold_cos"]].describe().T


## 5.0 Limpieza, imputación y tipología de variables

### 5.1 `PoolQC` y `GarageType`

Según `data/data_description.txt`, `NA` **no es un faltante aleatorio**:

| Variable | Significado de NA | Tratamiento |
| --- | --- | --- |
| `PoolQC` | *No Pool* | Categoría explícita `"None"` (nivel más bajo ordinal) |
| `GarageType` | *No Garage* | Categoría `"None"` (nominal adicional) |

Imputar con moda crearía albercas/garajes inventados. El relleno con `"None"` es **determinista** (no aprende de train), así que puede hacerse antes de partir sin *leakage*. La misma lógica aplica a Alley, Fence, FireplaceQu, Bsmt*, GarageQual/Cond/Finish, MiscFeature, MasVnrType.


In [ ]:
NA_AS_NONE: Final[list[str]] = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature", "MasVnrType",
]

print(
    "Faltantes antes -> PoolQC:",
    int(df["PoolQC"].isna().sum()),
    "| GarageType:",
    int(df["GarageType"].isna().sum()),
)
df[NA_AS_NONE] = df[NA_AS_NONE].fillna("None")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)  # sin revestimiento => 0

remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print("Faltantes restantes (se imputan dentro del Pipeline, solo con train):")
display(remaining.to_frame("n") if len(remaining) else pd.DataFrame({"n": []}))


### 5.2 Criterios de transformación

| Tipo | Imputación (*fit* solo en train) | Transformación | Por qué |
| --- | --- | --- | --- |
| Numéricas | Mediana + `add_indicator` | `StandardScaler` en lineales/SVM/MLP | Mediana robusta; el indicador conserva “no había dato” (p. ej. sin garaje) |
| Ordinales de calidad | Moda | `OrdinalEncoder` con orden explícito | Respeta Po < Fa < TA < Gd < Ex |
| Nominales | Moda | One-Hot (`drop` según modelo; §6) | Sin orden natural |
| Objetivo | — | `log1p` vía `TransformedTargetRegressor` | Reduce asimetría; métricas se reportan en USD (`expm1`) |


In [ ]:
QUAL_LEVELS: Final[list[str]] = ["None", "Po", "Fa", "TA", "Gd", "Ex"]

ORD_MAP: Final[dict[str, list[str]]] = {
    **{
        c: QUAL_LEVELS
        for c in [
            "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
            "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC",
        ]
    },
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "PavedDrive": ["N", "P", "Y"],
    "CentralAir": ["N", "Y"],
    "Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],
}

X: pd.DataFrame = df.drop(columns=TARGET_COL)
y: pd.Series = df[TARGET_COL]

ord_cols: list[str] = list(ORD_MAP)
for col in ord_cols:
    unseen = set(X[col].dropna().unique()) - set(ORD_MAP[col])
    assert not unseen, f"{col}: valores fuera de escala {unseen}"

nom_cols: list[str] = [
    c for c in X.select_dtypes(exclude="number").columns if c not in ord_cols
]
num_cols: list[str] = X.select_dtypes(include="number").columns.tolist()
print(f"{len(num_cols)} numéricas | {len(ord_cols)} ordinales | {len(nom_cols)} nominales")


## 6.0 One-Hot Encoding y el parámetro `drop`

- **No árboles** (OLS, Ridge/Lasso, SVR, MLP): `drop="first"` evita la trampa de dummies (colinealidad perfecta con el intercepto).
- **Árboles** (DT, RF, XGBoost): `drop=None`. No sufren la misma colinealidad; dropear una categoría obliga a aislarla como “todas las demás = 0”, profundiza el árbol y diluye importancias.


In [ ]:
def build_prep(*, tree: bool = False) -> ColumnTransformer:
    """
    tree=True  -> sin escalado, One-Hot completo.
    tree=False -> StandardScaler en num/ord + drop='first' en nominales.
    """
    scale: list[Any] = [] if tree else [StandardScaler()]
    num = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), *scale)
    ordi = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=[ORD_MAP[c] for c in ord_cols]),
        *scale,
    )
    nom = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(
            drop=None if tree else "first",
            handle_unknown="ignore",
            sparse_output=False,
        ),
    )
    return ColumnTransformer(
        [("num", num, num_cols), ("ord", ordi, ord_cols), ("nom", nom, nom_cols)]
    )


def make_estimator(model: Any, *, tree: bool = False) -> TransformedTargetRegressor:
    """Pipeline de features + modelo, con objetivo en log1p y predicción en USD."""
    pipe = Pipeline([("prep", build_prep(tree=tree)), ("model", model)])
    return TransformedTargetRegressor(
        regressor=pipe,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


## 7.0 Partición de los datos

Reservo **20 %** como *test* (no se toca hasta el final). Sobre el 80 % restante uso **validación cruzada de 5 folds** (K-Fold con shuffle): el conjunto es chico (~1,460 filas) y el enunciado permite partir solo en dos cuando hay CV.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED
)
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
print(f"train: {X_train.shape} | test: {X_test.shape}")


## 8.0 Modelo base (ingenuo)

Predice siempre la **media** (y, como referencia, la mediana) del precio en train. Cualquier modelo útil debe superar este piso con claridad.


In [ ]:
base_mean = DummyRegressor(strategy="mean").fit(X_train, y_train)
base_median = DummyRegressor(strategy="median").fit(X_train, y_train)

RMSE_BASE: float = rmse(y_test, base_mean.predict(X_test))
print(f"RMSE ingenuo (media):   ${RMSE_BASE:,.0f}")
print(f"RMSE ingenuo (mediana): ${rmse(y_test, base_median.predict(X_test)):,.0f}")
print(f"MAE  ingenuo (media):   ${mean_absolute_error(y_test, base_mean.predict(X_test)):,.0f}")


## 9.0 Entrenamiento y ajuste de hiperparámetros

Cada familia se ajusta por separado con búsqueda sobre **train** (CV, scoring RMSE). Luego:

- **CV_RMSE / CV_MAE**: predicciones *out-of-fold* en train → sirven para **elegir** modelos sin mirar test.
- **RMSE / MAE / R² / MAPE (test)**: desempeño final.
- **Tiempo**: búsqueda + ajuste del modelo final.
- **Curva de aprendizaje** en USD (gracias a `TransformedTargetRegressor`).


In [ ]:
RESULTS: list[dict[str, Any]] = []
BEST: dict[str, Any] = {}
FITTED: dict[str, Any] = {}


def plot_learning_curve(estimator: Any, name: str) -> None:
    sizes, tr, va = learning_curve(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=N_JOBS,
        shuffle=True,
        random_state=SEED,
    )
    tr, va = -tr, -va
    fig, ax = plt.subplots(figsize=(6.2, 4))
    ax.plot(sizes, tr.mean(1), "o-", color=PLOT_PRIMARY, label="Entrenamiento")
    ax.fill_between(
        sizes, tr.mean(1) - tr.std(1), tr.mean(1) + tr.std(1), alpha=0.15, color=PLOT_PRIMARY
    )
    ax.plot(sizes, va.mean(1), "o-", color=PLOT_SECONDARY, label="Validación (CV)")
    ax.fill_between(
        sizes, va.mean(1) - va.std(1), va.mean(1) + va.std(1), alpha=0.15, color=PLOT_SECONDARY
    )
    ymax = float(max(3 * np.nanmax(va.mean(1)), np.nanmax(tr.mean(1)) * 1.05))
    if np.isfinite(ymax) and ymax > 0:
        ax.set_ylim(0, ymax)
    style_axes(ax, f"Curva de aprendizaje — {name}", "Tamaño de entrenamiento", "RMSE (USD)")
    ax.legend()
    finalize_figure()


def add_result(
    name: str,
    best: Any,
    search_time: float = np.nan,
    params: Any = "",
    *,
    do_cv: bool = True,
) -> Any:
    final = clone(best)
    t0 = time.perf_counter()
    final.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0
    pred = final.predict(X_test)

    row: dict[str, Any] = {"Modelo": name}
    if do_cv:
        oof = cross_val_predict(clone(best), X_train, y_train, cv=cv, n_jobs=N_JOBS)
        row["CV_RMSE"] = rmse(y_train, oof)
        row["CV_MAE"] = float(mean_absolute_error(y_train, oof))
    else:
        row["CV_RMSE"] = np.nan
        row["CV_MAE"] = np.nan

    row.update(
        {
            "RMSE": rmse(y_test, pred),
            "MAE": float(mean_absolute_error(y_test, pred)),
            "R2": float(r2_score(y_test, pred)),
            "MAPE_%": float(mean_absolute_percentage_error(y_test, pred) * 100),
            "Tiempo_fit_s": fit_time,
            "Tiempo_busqueda_s": search_time,
            "Mejores_hiperparametros": str(params),
        }
    )
    RESULTS.append(row)
    BEST[name] = best
    FITTED[name] = final
    return final


def run_model(
    name: str,
    model: Any,
    params: dict[str, list[Any]] | None = None,
    *,
    tree: bool = False,
    search: str = "random",
    n_iter: int = 20,
) -> Any:
    est = make_estimator(model, tree=tree)
    t0 = time.perf_counter()
    best_params: dict[str, Any] = {}

    if params:
        grid = {f"regressor__model__{k}": v for k, v in params.items()}
        kw: dict[str, Any] = {
            "cv": cv,
            "scoring": "neg_root_mean_squared_error",
            "n_jobs": N_JOBS,
        }
        if search == "grid" and not QUICK:
            searcher: Any = GridSearchCV(est, grid, **kw)
        else:
            searcher = RandomizedSearchCV(
                est,
                grid,
                n_iter=2 if QUICK else n_iter,
                random_state=SEED,
                **kw,
            )
        searcher.fit(X_train, y_train)
        est = searcher.best_estimator_
        best_params = {
            k.replace("regressor__model__", ""): v
            for k, v in searcher.best_params_.items()
        }
        print(f"Mejores hiperparámetros: {best_params}")
        print(f"RMSE CV (búsqueda): ${-searcher.best_score_:,.0f}")

    search_time = time.perf_counter() - t0
    add_result(name, est, search_time, best_params)
    plot_learning_curve(est, name)
    display(
        pd.DataFrame(RESULTS)
        .set_index("Modelo")
        .loc[[name], ["CV_RMSE", "CV_MAE", "RMSE", "MAE", "R2", "MAPE_%", "Tiempo_fit_s"]]
        .round(3)
    )
    return FITTED[name]


### 9.1 Regresión lineal múltiple (OLS, Ridge, Lasso)


In [ ]:
run_model("OLS", LinearRegression())


In [ ]:
run_model(
    "Ridge",
    Ridge(random_state=SEED),
    {"alpha": [0.1, 0.3, 1, 3, 10, 30, 100, 300]},
    search="grid",
)


In [ ]:
run_model(
    "Lasso",
    Lasso(max_iter=20_000, random_state=SEED),
    {"alpha": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]},
    search="grid",
)


### 9.2 Árbol de decisión


In [ ]:
run_model(
    "Arbol",
    DecisionTreeRegressor(random_state=SEED),
    {
        "max_depth": [3, 4, 5, 6, 8, 10, 12, None],
        "min_samples_leaf": [1, 3, 5, 10, 20],
        "min_samples_split": [2, 5, 10, 20],
        "ccp_alpha": [0, 1e-5, 1e-4, 1e-3],
    },
    tree=True,
    n_iter=30,
)


### 9.3 Bosque aleatorio


In [ ]:
run_model(
    "RF",
    RandomForestRegressor(random_state=SEED, n_jobs=1),
    {
        "n_estimators": [200, 400],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.3, 0.5],
    },
    tree=True,
    n_iter=12,
)


### 9.4 XGBoost


In [ ]:
run_model(
    "XGB",
    XGBRegressor(tree_method="hist", random_state=SEED, n_jobs=1),
    {
        "n_estimators": [300, 600, 1000],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [2, 3, 4, 5],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.4, 0.6, 0.8],
        "min_child_weight": [1, 3, 5],
        "reg_lambda": [0.5, 1, 5, 10],
    },
    tree=True,
    n_iter=25,
)


### 9.5 Perceptrón multicapa (MLP)


In [ ]:
run_model(
    "MLP",
    MLPRegressor(
        solver="adam",
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        max_iter=1000,
        random_state=SEED,
    ),
    {
        "hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
        "alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1],
        "learning_rate_init": [1e-3, 3e-3, 1e-2],
        "activation": ["relu", "tanh"],
    },
    n_iter=15,
)


### 9.6 Máquina de vector soporte (SVR)


In [ ]:
run_model(
    "SVR",
    SVR(kernel="rbf"),
    {
        "C": [0.3, 1, 3, 10, 30, 100],
        "epsilon": [0.01, 0.03, 0.1],
        "gamma": ["scale", 0.0003, 0.001, 0.003],
    },
    n_iter=20,
)


### 9.7 Tabla resumen de modelos individuales


In [ ]:
res = pd.DataFrame(RESULTS)
cols = [
    "Modelo", "CV_RMSE", "CV_MAE", "RMSE", "MAE", "R2", "MAPE_%",
    "Tiempo_fit_s", "Tiempo_busqueda_s",
]
print(f"Referencia — RMSE ingenuo en test: ${RMSE_BASE:,.0f}")
display(res[cols].sort_values("CV_RMSE").round(3).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = res.sort_values("CV_RMSE")
ax.barh(plot_df["Modelo"], plot_df["CV_RMSE"], color=PLOT_PRIMARY, edgecolor=PLOT_EDGE, label="CV RMSE")
ax.barh(plot_df["Modelo"], plot_df["CV_MAE"], color=PLOT_SECONDARY, alpha=0.75, edgecolor=PLOT_EDGE, label="CV MAE")
style_axes(ax, "Error out-of-fold en train", "USD", "")
ax.legend()
finalize_figure()


**Lectura de curvas (sesión de modelado).** En general espero:

- **Árbol** con brecha train/validación más marcada (varianza).
- **Ridge / Lasso / SVR / XGB** con validación más cercana al train a medida que crece el tamaño.
- **MLP** más sensible a escala e inicialización; si la curva de validación se estanca alto, más datos o más regularización ayudarían.
- Coste: XGB/RF/MLP/Stacking pesan más en búsqueda; Ridge/Lasso/SVR (n≈1k) son baratos de reentrenar.


## 10.0 Análisis de factores del mejor modelo

Uso **importancia por permutación** sobre test, con el pipeline completo: el aumento de RMSE (USD) al permutar una variable **original** (no cada dummy).


In [ ]:
best_name: str = pd.DataFrame(RESULTS).sort_values("CV_RMSE").iloc[0]["Modelo"]
print("Mejor modelo individual (menor CV_RMSE):", best_name)

pi = permutation_importance(
    FITTED[best_name],
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=SEED,
    n_jobs=N_JOBS,
)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
top = imp.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(top.index, top.values, color=PLOT_ACCENT, edgecolor=PLOT_EDGE)
style_axes(
    ax,
    f"Importancia por permutación — {best_name}",
    "Aumento del RMSE (USD) al permutar",
    "",
)
finalize_figure()
display(imp.head(15).round(0).to_frame("delta_RMSE_USD"))


In [ ]:
top_num = [c for c in imp.index if c in num_cols][:4]
X_pd = X_train.copy()
X_pd[top_num] = X_pd[top_num].astype(float)
PartialDependenceDisplay.from_estimator(
    FITTED[best_name], X_pd, top_num, grid_resolution=30, n_jobs=1
)
plt.suptitle("Dependencia parcial (precio predicho vs variable)")
plt.tight_layout()
plt.show()


### Interpretación en contexto inmobiliario

En Ames suelen dominar, y eso es coherente con un valuador humano:

1. **`OverallQual` / calidades** (`KitchenQual`, `ExterQual`, `BsmtQual`): acabados y percepción de calidad.
2. **Área habitable** (`GrLivArea`, `TotalBsmtSF`, baños): más metros útiles, más precio (con rendimientos decrecientes).
3. **Garaje** (`GarageCars` / `GarageArea`) y **chimenea**.
4. **Ubicación** (`Neighborhood_*`): externalidades de zona.
5. **Edades** (`AgeHouse`, `AgeRemod`): más antigua o sin remodelar → presión a la baja.
6. **Estacionalidad** (`MoSold_sin/cos`): efecto chico frente a calidad y tamaño.

Limitación: la importancia por permutación se **reparte** entre variables correlacionadas y mide asociación, no causalidad.


## 11.0 Tres mejores modelos con RMSE + MAE

La suma se calcula con métricas **CV (OOF)** para no usar test en la selección.


In [ ]:
res = pd.DataFrame(RESULTS)
res["CV_RMSE+MAE"] = res["CV_RMSE"] + res["CV_MAE"]
rank = (
    res.sort_values("CV_RMSE+MAE")[
        ["Modelo", "CV_RMSE", "CV_MAE", "CV_RMSE+MAE", "RMSE", "MAE"]
    ]
    .round(0)
    .reset_index(drop=True)
)
top3: list[str] = rank["Modelo"].head(3).tolist()
print("Tres mejores modelos:", top3)
display(rank)


### ¿Qué aporta RMSE + MAE?

**Ventajas (valuación):** el MAE es el error típico en dólares (fácil de explicar); el RMSE castiga fallos grandes (sub/sobrevaluar una casa cara duele más). La suma evita elegirlo solo por un extremo.

**Desventajas:** el peso 1:1 es arbitrario; RMSE ≥ MAE y suele dominar; no es métrica “de libro”; está en dólares absolutos (casas caras pesan más) y no mete costo computacional ni interpretabilidad.


## 12.0 Ensambles: Voting y Stacking

**`VotingRegressor`:** promedia predicciones de bases independientes. Simple, baja varianza si los errores no están perfectamente correlacionados.

**`StackingRegressor`:** un metamodelo (`RidgeCV` sobre predicciones OOF de las bases) aprende cómo combinarlas. Suele ganar un poco más de precisión a costa de tiempo y complejidad.


### 12.1 VotingRegressor


In [ ]:
voting = VotingRegressor([(n, clone(BEST[n])) for n in top3], n_jobs=N_JOBS)
add_result("Voting", voting, do_cv=False)
display(
    pd.DataFrame(RESULTS)
    .set_index("Modelo")
    .loc[["Voting"], ["RMSE", "MAE", "R2", "MAPE_%", "Tiempo_fit_s"]]
    .round(3)
)


### 12.2 StackingRegressor


In [ ]:
stacking = StackingRegressor(
    [(n, clone(BEST[n])) for n in top3],
    final_estimator=make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=np.logspace(-2, 3, 12)),
    ),
    cv=cv,
    n_jobs=N_JOBS,
)
add_result("Stacking", stacking, do_cv=False)

ridge = FITTED["Stacking"].final_estimator_[-1]
print("Pesos del metamodelo (Ridge):", dict(zip(top3, np.ravel(ridge.coef_).round(3))))
display(
    pd.DataFrame(RESULTS)
    .set_index("Modelo")
    .loc[["Stacking"], ["RMSE", "MAE", "R2", "MAPE_%", "Tiempo_fit_s"]]
    .round(3)
)


### 12.3 Comparación individuales vs ensambles


In [ ]:
res = pd.DataFrame(RESULTS)
comp = res[res["Modelo"].isin(top3 + ["Voting", "Stacking"])].sort_values("RMSE")
display(
    comp[["Modelo", "RMSE", "MAE", "R2", "MAPE_%", "Tiempo_fit_s"]]
    .round(3)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
sns.barplot(data=comp, x="RMSE", y="Modelo", ax=ax, color=PLOT_PRIMARY, edgecolor=PLOT_EDGE)
style_axes(ax, "RMSE en test", "USD", "")
finalize_figure()

best_single = comp[comp["Modelo"].isin(top3)].iloc[0]
for ens in ["Voting", "Stacking"]:
    row = comp[comp["Modelo"] == ens].iloc[0]
    delta = best_single["RMSE"] - row["RMSE"]
    speed = row["Tiempo_fit_s"] / max(best_single["Tiempo_fit_s"], 1e-9)
    print(
        f"{ens}: RMSE ${row['RMSE']:,.0f} vs mejor individual "
        f"({best_single['Modelo']}) ${best_single['RMSE']:,.0f} "
        f"→ Δ ${delta:,.0f} ({delta / best_single['RMSE'] * 100:.1f} %), "
        f"tiempo ×{speed:.1f}"
    )


**¿Conviene el ensamble?** Si la mejora en test es clara y estable (>~1–2 % de RMSE) y el pipeline es batch, Voting/Stacking se justifican. Si la ganancia es marginal, priorizo el mejor individual: menos piezas que versionar, más fácil de explicar a negocio y de monitorear.


## 13.0 Reportes finales

Cifras que alimentan ambos reportes (rellenar la narrativa con los números de *esta* corrida):


In [ ]:
res = pd.DataFrame(RESULTS)
summary = (
    res[["Modelo", "RMSE", "MAE", "R2", "MAPE_%", "Tiempo_fit_s"]]
    .sort_values("RMSE")
    .round(2)
    .reset_index(drop=True)
)
display(summary)
best_row = summary.iloc[0]
print(f"Modelo ingenuo: RMSE ${RMSE_BASE:,.0f}")
print(
    f"Mejor en test: {best_row['Modelo']} → RMSE ${best_row['RMSE']:,.0f}, "
    f"MAE ${best_row['MAE']:,.0f}, R² {best_row['R2']:.3f}, MAPE {best_row['MAPE_%']:.1f} %"
)
print(
    f"Reducción del error vs ingenuo: {(1 - best_row['RMSE'] / RMSE_BASE) * 100:.0f} %"
)
print("Top-3 por CV RMSE+MAE:", top3)


### 13.1 Reporte técnico (equipo de datos / ML)

**Recomendación:** desplegar el mejor modelo de *esta* corrida según RMSE/MAE en test, con el preprocesamiento versionado en un único `Pipeline` (+ `TransformedTargetRegressor`).

1. **Métricas.** Comparar RMSE/MAE/R²/MAPE de test contra el ingenuo; verificar que el orden del top-3 por CV (RMSE+MAE) sea estable en test.
2. **Sesgo–varianza.** Revisar curvas de aprendizaje: si la validación sigue bajando al final, más datos ayudan; si hay brecha grande (p. ej. árbol), regularizar o ensamblar.
3. **Ensambles.** Adoptar Voting/Stacking solo si la mejora compensa ×tiempo y la pérdida de interpretabilidad.
4. **Operación.** Reentrenar periódico (p. ej. trimestral); recalcular edades con el año corriente; **no** reintroducir `YrSold` si se mantiene el supuesto estático; monitorear drift por vecindario y por rango de precio.
5. **Riesgos.** Datos de Ames (2006–2010): recalibrar antes de otro mercado; errores absolutos crecen en casas caras; MLP suele ser el eslabón débil con este tamaño de muestra.


### 13.2 Reporte ejecutivo (dirección inmobiliaria)

**En una frase:** el sistema estima el precio de una casa en segundos y reduce de forma clara el error frente a usar solo el precio promedio histórico.

- **Beneficios.** Valuaciones consistentes, apoyo a listados/negociaciones y señales de qué atributos mueven el precio (calidad, metros, garaje, zona).
- **Riesgos.** No sustituye al valuador en casos atípicos o de lujo; fue entrenado en otra ciudad/época; el mercado puede cambiar; conviene un margen de seguridad en decisiones de alto monto.
- **Costos.** Cómputo bajo a esta escala; el costo real es mantenimiento de datos limpios y revisión periódica del error.
- **Decisión sugerida.** Piloto en paralelo con valuadores, comparar estimaciones vs cierres reales y decidir ampliación con evidencia.


## 14.0 Conclusiones


Incluye todos los comentarios y conclusiones de esta actividad. En particular, responde:

- ¿En qué casos el uso de LLMs fue de mayor provecho?
- ¿En cuáles casos tu intervención como experto en combinación con los LLM fue importante?


### +++++++++++ Inicia sección de Conclusiones Finales ++++++++++++++++

**Conclusiones técnicas**

- Cumplir las restricciones temporales (`drop YrSold`, edades a 2026, `MoSold` cíclico) alinea el problema con una valuación comparable entre transacciones, no con un forecast de mercado.
- Tratar `NA` en `PoolQC` / `GarageType` (y afines) como **ausencia** evita inventar amenidades; la imputación estadística vive dentro del `Pipeline` para no filtrar información.
- Modelar `log1p(SalePrice)` y reportar en dólares estabiliza el aprendizaje sin confundir al negocio.
- En este tamaño de datos, lineales regularizados, SVR y boosting suelen concentrar el mejor equilibrio; el árbol solo y el MLP (con presupuesto de búsqueda moderado) quedan atrás.
- RMSE + MAE es un criterio práctico de selección; Voting/Stacking aportan solo si la mejora no es cosmética.

**Uso de IA generativa (LLMs)**

Herramientas: **Claude** (Anthropic) para el primer esqueleto metodológico del cuaderno, y **Cursor / Composer** para alinear el código al estilo del repositorio del curso, rutas de datos y endurecimiento anti-*leakage*.

_¿En qué casos fue de mayor provecho?_

- Armar la estructura completa contra la rúbrica (17 puntos) y no omitir entregables.
- Scaffolding de `Pipeline` / `ColumnTransformer` / `TransformedTargetRegressor` y tablas de métricas.
- Borradores de justificación (temporales, OHE `drop`, reportes técnico/ejecutivo).

_¿En qué casos fue importante la intervención como experto?_

- Verificar el diccionario de datos (`NA` = ausencia, no MCAR).
- Separar preprocesamiento determinista vs. estadísticos que deben ajustarse solo en train/folds.
- Elegir MoSold cíclico, CV 5-fold en lugar de un val fijo, y metamodelo escalado en Stacking.
- Traducir métricas a decisión de negocio (¿ensamble o modelo simple?) y mantener el tono/estructura del resto de mis entregas del curso.

_Limitación:_ un LLM puede afirmar cifras que no salieron de la corrida; todas las métricas de los reportes deben cuadrar con las celdas ejecutadas de *esta* sesión.

### +++++++++++ Termina sección de conclusiones finales ++++++++++++++


## 15.0 Anexo — prompts


**Incluir a continuación los prompts más importantes y el nombre del modelo LLM utilizado.**


### +++++++++++ Inicia sección de prompts ++++++++++++++++

**LLMs utilizados:** Claude (Anthropic); Composer en Cursor.

**Prompt 1:** Construir el cuaderno completo de la actividad de predicción de precios de vivienda según el PDF, con las restricciones de `YrSold`, edades a 2026 y decisión sobre `MoSold`, usando el `train.csv` de Kaggle.

**Prompt 2:** Revisar el cuaderno contra buenas prácticas (evitar *data leakage*, `drop` en One-Hot según familia de modelo, CV + hold-out, Voting/Stacking) y dejarlo ejecutable.

**Prompt 3:** Reescribir el cuaderno con el estándar de mis entregas del repo `Inteligencia-artificial-y-aprendizaje-autom-tico` (encabezado con nombre/matrícula, secciones `1.0`, constantes tipadas, helpers con docstring, conclusiones en el mismo tono).

### +++++++++++ Termina sección de prompts ++++++++++++++


# **<<< Fin de la Actividad — Predicción de Precios de Casas >>>**
